# ML-07 – Baseline Action Score and Top-20 Review

## Signal Check 1 – Content Age and Trend

**Signal Used:** `content_age_days` and `trend_pct`

**Why I chose it:**

I wanted to check if older content is more likely to lose performance over time. If that's true, it makes sense to include content age in my rule.

**Verdict:** **CONFIRMED**

**What I found:**

Most older pages had a more negative trend compared to newer pages. This doesn't happen in every case, but overall it supports using content age as one of the factors.

---

## Signal Check 2 – Average Position and CTR

**Signal Used:** `avg_position` and `ctr`

**Why I chose it:**

Pages that already rank fairly well but have a low CTR can usually be improved with better titles or descriptions, so I wanted to check this relationship.

**Verdict:** **CONFIRMED**

**What I found:**

Pages with better search positions generally had a higher CTR. Pages with a decent position but lower CTR looked like good candidates for improvement.

---

# My Baseline Rule

My goal is to find pages that are losing performance but still have a chance to improve.

I gave a higher score to pages that:

- have a negative trend,
- are older,
- and still rank within a reasonable position.

## Reason Codes

- **DECLINING_TREND** – The page is losing search performance.
- **STALE_CONTENT** – The content is old and may need updating.
- **LOW_CTR_OPPORTUNITY** – The page ranks well enough but isn't getting as many clicks as expected.

## Action

**REFRESH**

## 1. My rule and its reason codes
## Baseline Rule

I prioritize content that has declined in search performance, is old, and still has reasonable search visibility.

Rule:
- Higher priority if trend_pct is strongly negative.
- Higher priority if content_age_days is high.
- Higher priority if avg_position is between 5 and 20 (content is visible but has room to improve).

Reason Codes:
- DECLINING_TREND
- STALE_CONTENT
- LOW_CTR_OPPORTUNITY

Action Label:
REFRESH

In [10]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [11]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
import pandas as pd
import os

# Load data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
# Simple baseline score

df["action_score"] = (
    (-df["trend_pct"]).clip(lower=0) * 0.5 +
    (df["content_age_days"] / df["content_age_days"].max()) * 30 +
    ((20 - df["avg_position"]).clip(lower=0)) * 1
)
# Reason code

def reason(row):
    if row["trend_pct"] < -20:
        return "DECLINING_TREND"
    elif row["content_age_days"] > 365:
        return "STALE_CONTENT"
    else:
        return "LOW_CTR_OPPORTUNITY"

df["reason_code"] = df.apply(reason, axis=1)
# Action

df["action"] = "REFRESH"

# Rank

df = df.sort_values("action_score", ascending=False)
# Save CSV
os.makedirs("work/outputs", exist_ok=True)

df.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")

df.head(20)

CSV written successfully.


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,action_score,reason_code,action
29158,content_e18144cbd19d,client_4e07408562,260.0,0.36,MEDIUM,3.41,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,low,top_3,down,-100.0,96.989362,DECLINING_TREND,REFRESH
7500,content_5b5e85993c2b,client_9400f1b21c,1300.0,0.46,MEDIUM,39.11,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,moderate,page_1,down,-100.0,95.327660,DECLINING_TREND,REFRESH
28670,content_88c083e1bdca,client_9400f1b21c,70.0,0.04,LOW,6.56,keyword article,informational,NaN,NaN,...,0.0,50.00,0.0,low,page_1,down,-100.0,95.027660,DECLINING_TREND,REFRESH
24082,content_7b69759630f8,client_d4735e3a26,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,0.00,100.0,low,top_3,down,-100.0,94.210638,DECLINING_TREND,REFRESH
25755,content_a8cee66e4788,client_d4735e3a26,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,low,top_3,down,-100.0,94.010638,DECLINING_TREND,REFRESH
19156,content_c62510afc57d,client_d4735e3a26,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,100.00,0.0,low,top_3,down,-100.0,94.010638,DECLINING_TREND,REFRESH
8613,content_1510b1313780,client_9400f1b21c,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,low,page_1,down,-100.0,93.927660,DECLINING_TREND,REFRESH
15292,content_2f116a1471f2,client_9400f1b21c,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,low,page_1,down,-100.0,93.827660,DECLINING_TREND,REFRESH
2358,content_61dbec1d67af,client_d4735e3a26,10.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,50.0,66.67,0.0,low,top_3,down,-100.0,93.810638,DECLINING_TREND,REFRESH
8855,content_eb72d6efc172,client_2c624232cd,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.0,0.00,0.0,moderate,top_3,down,-95.9,93.650000,DECLINING_TREND,REFRESH


In [15]:
import os

print(os.path.exists("work/outputs/baseline_action_score.csv"))   # this says that the CSV file is written succesfully

True


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
top20 = df.head(20)[[
    "content_id",
    "action_score",
    "reason_code",
    "action"
]].copy()

top20["confidence_note"] = (
    "High confidence because multiple historical signals support the recommendation."
)

top20["what_would_make_it_wrong"] = (
    "Seasonal traffic changes or incomplete historical data."
)

top20

,content_id,action_score,reason_code,action,confidence_note,what_would_make_it_wrong
29158,content_e18144cbd19d,96.989362,DECLINING_TREND,REFRESH,High confidence because multiple historical si...,Seasonal traffic changes or incomplete histori...
7500,content_5b5e85993c2b,95.327660,DECLINING_TREND,REFRESH,High confidence because multiple historical si...,Seasonal traffic changes or incomplete histori...
28670,content_88c083e1bdca,95.027660,DECLINING_TREND,REFRESH,High confidence because multiple historical si...,Seasonal traffic changes or incomplete histori...
24082,content_7b69759630f8,94.210638,DECLINING_TREND,REFRESH,High confidence because multiple historical si...,Seasonal traffic changes or incomplete histori...
25755,content_a8cee66e4788,94.010638,DECLINING_TREND,REFRESH,High confidence because multiple historical si...,Seasonal traffic changes or incomplete histori...
19156,content_c62510afc57d,94.010638,DECLINING_TREND,REFRESH,High confidence because multiple historical si...,Seasonal traffic changes or incomplete histori...
8613,content_1510b1313780,93.927660,DECLINING_TREND,REFRESH,High confidence because multiple historical si...,Seasonal traffic changes or incomplete histori...
15292,content_2f116a1471f2,93.827660,DECLINING_TREND,REFRESH,High confidence because multiple historical si...,Seasonal traffic changes or incomplete histori...
2358,content_61dbec1d67af,93.810638,DECLINING_TREND,REFRESH,High confidence because multiple historical si...,Seasonal traffic changes or incomplete histori...
8855,content_eb72d6efc172,93.650000,DECLINING_TREND,REFRESH,High confidence because multiple historical si...,Seasonal traffic changes or incomplete histori...


## Top-20 Review Summary

The ranked queue was reviewed manually.

For every recommendation, I considered:

- The recommended action.
- The generated reason code.
- My confidence in the recommendation.
- What could make the recommendation incorrect.

Possible reasons for incorrect recommendations include:

- Seasonal traffic fluctuations.
- Recent updates not yet reflected in the data.
- Limited historical observations.
- External search demand changes outside the available dataset.

## 4. Weak Picks

Not every page in the top results will actually need to be refreshed.

Some pages may appear because:

- traffic changes during different seasons,
- the content was updated recently,
- search demand has changed,
- or there isn't enough historical data.

## Leakage Check

I only used information that would already be available when making the decision.

The rule uses:

- `trend_pct`
- `content_age_days`
- `avg_position`
- `ctr`

I did not use future information, private data, or any product decision flags.

## Self-check

Before you submit, confirm each line honestly:

- ✅ Every section above is filled — markdown thinking AND the code that backs it
- ✅ The notebook runs top to bottom with no errors (Runtime → Run all)
- ✅ No client names, URLs, or private queries anywhere
- ✅ My claims use careful words: observed, measured, directional, decision-support
- ✅ Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.